In [1]:
import customtkinter as ctk
from tkinter import filedialog
import os
import threading
import time # También asegúrate de tener este, ya que lo usaremos para las esperas
import serial
import serial.tools.list_ports # Opcional: útil para listar los COM disponibles
import ezdxf
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# ==========================================
# CONEXIÓN SERIAL ESP32
# ==========================================

try:

    ser = serial.Serial('COM6',115200,timeout=0.1)
    time.sleep(2)
    ser.reset_input_buffer()
    print("ESP32 conectada por Serial")

except Exception as e:

    ser = None
    print("No se pudo conectar con ESP32")
    print(e)
    
class InterfazCADCompleta(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.title("Interfaz de Control")
        self.geometry("1000x600") # Ventana más grande para la vista previa
        ctk.set_appearance_mode("dark")

        # Configuración del Grid (Menú, Info, Vista Previa)
        self.grid_columnconfigure(1, weight=1) # Columna central (Info)
        self.grid_columnconfigure(2, weight=3) # Columna derecha (Vista Previa)
        self.grid_rowconfigure(0, weight=1)

        # Variables de estado
        self.ruta_cad = ""
        self.coordenadas = [] # Aquí guardaremos los puntos [x, y]

        # ==========================================
        # --- COLUMNA 0: MENÚ LATERAL ---
        # ==========================================
        self.menu_lateral = ctk.CTkFrame(self, width=200, corner_radius=0)
        self.menu_lateral.grid(row=0, column=0, sticky="nsew")
        
        self.lbl_titulo = ctk.CTkLabel(self.menu_lateral, text="ROBOT NAV", font=("Arial", 20, "bold"))
        self.lbl_titulo.pack(pady=20, padx=10)

        self.btn_manual = ctk.CTkButton(self.menu_lateral, text="Modo Manual", 
                                        command=lambda: print("Modo Manual Activo"))
        self.btn_manual.pack(pady=10, padx=20)

        # BOTÓN DE CARGA
        self.btn_cad = ctk.CTkButton(self.menu_lateral, text="Cargar CAD (DXF)", 
                                     fg_color="cyan", text_color="black",
                                     command=self.procesar_cad)
        self.btn_cad.pack(pady=20, padx=20)

        self.lbl_espacio = ctk.CTkLabel(self.menu_lateral, text="")
        self.lbl_espacio.pack(expand=True)

        # BOTÓN DE MARCHA
        self.btn_marcha = ctk.CTkButton(self.menu_lateral, text="Iniciar Trayectoria", 
                                     fg_color="cyan", text_color="black",
                                     command=self.iniciar_trayectoria)
        self.btn_marcha.pack(pady=20, padx=20)

        self.lbl_espacio = ctk.CTkLabel(self.menu_lateral, text="")
        self.lbl_espacio.pack(expand=True)
        # BOTÓN DE EMERGENCIA
        self.btn_emergencia = ctk.CTkButton(self.menu_lateral, text="PARADA DE EMERGENCIA", 
                                            fg_color="red", hover_color="#8B0000",
                                            command=lambda: print("¡EMERGENCIA!"))
        self.btn_emergencia.pack(pady=20, padx=20)

        # ==========================================
        # --- COLUMNA 1: INFORMACIÓN DEL ARCHIVO ---
        # ==========================================
        self.frame_info = ctk.CTkFrame(self, fg_color="transparent")
        self.frame_info.grid(row=0, column=1, padx=20, pady=20, sticky="nsew")

        self.lbl_bienvenida = ctk.CTkLabel(self.frame_info, text="Gestión de Archivos", font=("Arial", 24))
        self.lbl_bienvenida.pack(pady=10)

        self.lbl_archivo = ctk.CTkLabel(self.frame_info, text="Archivo: Ninguno", 
                                        font=("Arial", 14), text_color="gray")
        self.lbl_archivo.pack(pady=10)

        # Cuadro de texto para log de coordenadas
        self.lbl_estado = ctk.CTkLabel(self.frame_info, text="Estado: Sistema Listo", 
                                       font=("Arial", 16), text_color="green")
        self.lbl_estado.pack(pady=10)
        self.txt_log = ctk.CTkTextbox(self.frame_info, width=250, height=250)
        self.txt_log.pack(pady=20)
        self.txt_log.insert("0.0", "Esperando archivo DXF...\n")

        self.lbl_puntos = ctk.CTkLabel(self.frame_info, text="Puntos extraídos: 0", 
                                        font=("Arial", 12))
        self.lbl_puntos.pack(pady=10)

        #Factor de Escala
        self.lbl_escala = ctk.CTkLabel(self.frame_info, text="Escala: 1 unidad CAD = ... mm", font=("Arial", 12))
        self.lbl_escala.pack(pady=(10, 0))
        
        self.entry_escala = ctk.CTkEntry(self.frame_info, placeholder_text="Ej: 10")
        self.entry_escala.insert(0, "1") # Valor inicial sugerido
        self.entry_escala.pack(pady=5)
        # ==========================================
        # --- COLUMNA 2: VISTA PREVIA (MATPLOTLIB) ---
        # ==========================================
        self.frame_preview = ctk.CTkFrame(self)
        self.frame_preview.grid(row=0, column=2, padx=20, pady=20, sticky="nsew")
        
        self.lbl_preview = ctk.CTkLabel(self.frame_preview, text="Vista Previa del Plano", font=("Arial", 18))
        self.lbl_preview.pack(pady=10)

        # Crear la figura de Matplotlib
        self.fig, self.ax = plt.subplots(figsize=(5, 4))
        self.fig.patch.set_facecolor('#2B2B2B') # Color de fondo oscuro de CTk
        self.ax.set_facecolor('#2B2B2B')
        self.ax.tick_params(colors='white') # Ejes en blanco
        self.ax.spines['bottom'].set_color('white')
        self.ax.spines['left'].set_color('white')
        
        # Integrar la figura en CustomTkinter
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.frame_preview)
        self.canvas.get_tk_widget().pack(fill="both", expand=True, padx=10, pady=10)

    # ==========================================
    # --- LÓGICA DE PROCESAMIENTO DXF ---
    # ==========================================
    def procesar_cad(self):
        # 1. Abrir explorador (filtrando solo por DXF)
        archivo = filedialog.askopenfilename(
            title="Seleccionar archivo DXF R12",
            filetypes=(("Archivos DXF", "*.dxf"),)
        )
        
        if not archivo:
            return # Cancelado
            
        self.ruta_cad = archivo
        nombre_archivo = archivo.split("/")[-1]
        self.lbl_archivo.configure(text=f"Archivo: {nombre_archivo}", text_color="cyan")
        self.txt_log.delete("0.0", "end")
        self.txt_log.insert("0.0", f"Leyendo: {nombre_archivo}...\n")
        
        # Iniciar las coordenadas de cero
        self.coordenadas = []

        try:
            # 2. Leer el DXF con ezdxf
            doc = ezdxf.readfile(self.ruta_cad)
            msp = doc.modelspace() # Accedemos al espacio de modelo (donde está el dibujo)

            # 3. Iterar sobre las entidades (LÍNEAS y POLILÍNEAS)
            for entity in msp.query('LINE LWPOLYLINE'):
                if entity.dxftype() == 'LINE':
                    # Una línea tiene punto de inicio (start) y fin (end)
                    start = entity.dxf.start
                    end = entity.dxf.end
                    # Guardamos la trayectoria: [x_inicio, y_inicio, x_fin, y_fin]
                    self.coordenadas.append([start.x, start.y, end.x, end.y])
                    self.txt_log.insert("end", f"LÍNEA: ({start.x:.1f},{start.y:.1f}) -> ({end.x:.1f},{end.y:.1f})\n")
                
                elif entity.dxftype() == 'LWPOLYLINE':
                    # Una polilínea es una secuencia de puntos conectados
                    puntos = list(entity.vertices())
                    for i in range(len(puntos) - 1):
                        p1 = puntos[i]
                        p2 = puntos[i+1]
                        self.coordenadas.append([p1[0], p1[1], p2[0], p2[1]])
                        self.txt_log.insert("end", f"POLI: ({p1[0]:.1f},{p1[1]:.1f}) -> ({p2[0]:.1f},{p2[1]:.1f})\n")

            self.lbl_puntos.configure(text=f"Trayectorias extraídas: {len(self.coordenadas)}")
            self.txt_log.insert("end", "Carga completa.\n")
            self.txt_log.see("end")

            # 4. Actualizar la Vista Previa
            self.actualizar_vista_previa()

        except Exception as e:
            self.txt_log.insert("end", f"ERROR: No se pudo leer el archivo. Asegúrate de que sea un DXF válido (R12).\nDetalle: {e}\n")
            print(f"Error ezdxf: {e}")

    # ==========================================
    # --- ACTUALIZAR GRÁFICO (VISTA PREVIA) ---
    # ==========================================
    def actualizar_vista_previa(self):
        # Limpiar el gráfico anterior
        self.ax.clear()
        self.ax.set_facecolor('#2B2B2B')
        self.ax.tick_params(colors='white')
        self.ax.set_title(f"Plano: {os.path.basename(self.ruta_cad)}", color='white')

        # Dibujar cada trayectoria
        for tray in self.coordenadas:
            # tray = [x_inicio, y_inicio, x_fin, y_fin]
            x_values = [tray[0], tray[2]]
            y_values = [tray[1], tray[3]]
            self.ax.plot(x_values, y_values, color='cyan', linewidth=1.5)

        # Asegurar que la escala sea igual (que no se deforme el dibujo)
        self.ax.set_aspect('equal', adjustable='box')
        self.fig.tight_layout()
        
        # Refrescar el canvas dentro de la interfaz
        self.canvas.draw()

    def on_closing(self):
        plt.close('all') # Cerrar figuras de matplotlib para liberar memoria
        self.destroy()
    
    def resaltar_punto_actual(self, x, y):
        # Dibujamos un punto rojo sobre el gráfico existente
        self.ax.plot(x, y, 'ro', markersize=5) 
        self.canvas.draw()
        
    def actualizar_consola(self, mensaje):
        self.txt_log.insert("end", f"{mensaje}\n")
        self.txt_log.see("end") # Hace scroll automático hacia abajo
        
    def cambiar_estado(self, mensaje):
        # Esta es la que te faltaba
        self.lbl_estado.configure(text=mensaje)
        if "EMERGENCIA" in mensaje:
            self.lbl_estado.configure(text_color="red")
        elif "EJECUTANDO" in mensaje:
            self.lbl_estado.configure(text_color="yellow")
        else:
            self.lbl_estado.configure(text_color="cyan")
        print(f"Estado: {mensaje}")

    def actualizar_consola(self, mensaje):
        # Esta la llama secuencia_movimiento para mostrar info en el cuadro de texto
        self.txt_log.insert("end", f"{mensaje}\n")
        self.txt_log.see("end")

    def leer_serial(self):
        # Opcional: para leer lo que la ESP32 te responde (ACK)
        while True:
            if ser and ser.is_open and ser.in_waiting > 0:
                linea = ser.readline().decode('utf-8').strip()
                self.actualizar_consola(f"ESP32: {linea}")
            time.sleep(0.01)
    # Dentro de tu clase InterfazCADCompleta...
    
    def iniciar_trayectoria(self):
        try:
            # Obtenemos la escala que el usuario escribió (ej: 10 si 1u = 10mm)
            factor = float(self.entry_escala.get())
        except ValueError:
            self.cambiar_estado("Error: Escala no válida")
            return
    
        threading.Thread(target=self.ejecutar_puntos, args=(factor,), daemon=True).start()
    
    def ejecutar_puntos(self, factor):

        for i, tray in enumerate(self.coordenadas):
    
            # =====================================
            # ESCALA CAD -> MM
            # =====================================
    
            x_mm = tray[2] * factor
            y_mm = tray[3] * factor
    
            comando = f"GOTO:{x_mm:.2f},{y_mm:.2f}\n"
    
            # =====================================
            # ENVIAR
            # =====================================
    
            if ser and ser.is_open:
    
                ser.reset_input_buffer()
    
                ser.write(comando.encode())
    
                self.actualizar_consola(
                    f"Enviando: {comando.strip()}"
                )
    
                # =================================
                # ESPERAR ACK
                # =================================
    
                timeout = time.time() + 15
    
                while time.time() < timeout:
    
                    if ser.in_waiting > 0:
    
                        respuesta = (
                            ser.readline()
                            .decode("utf-8")
                            .strip()
                        )
    
                        self.actualizar_consola(
                            f"ESP32: {respuesta}"
                        )
    
                        if respuesta == "ACK":
    
                            break
    
                    time.sleep(0.01)
    
            else:
    
                self.actualizar_consola(
                    f"Simulando: {comando.strip()}"
                )
    
            # =====================================
            # RESALTAR PUNTO
            # =====================================
    
            self.resaltar_punto_actual(x_mm, y_mm)
    
        self.cambiar_estado("TRAYECTORIA FINALIZADA")
# Ejecución
if __name__ == "__main__":
    app = InterfazCADCompleta()
    app.protocol("WM_DELETE_WINDOW", app.on_closing)
    app.mainloop()

No se pudo conectar con ESP32
could not open port 'COM6': FileNotFoundError(2, 'El sistema no puede encontrar el archivo especificado.', None, 2)
